# OpenAI Agents SDK

In [22]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
import asyncio


In [4]:
load_dotenv(override=True)

True

In [9]:
agent = Agent(name="Jokester", instructions="You are a jokester. Tell jokes to make people laugh.", model="gpt-4")

In [10]:
result = Runner.run(agent, "Tell me a joke.about AI agents and programming.")
print(result)


<coroutine object Runner.run at 0x0000026379C98640>


C:\Users\thiru\AppData\Local\Temp\ipykernel_31000\2093412423.py:1: RuntimeWarning: coroutine 'Runner.run' was never awaited
  result = Runner.run(agent, "Tell me a joke.about AI agents and programming.")


In [11]:
result = await Runner.run(agent, "Tell me a joke about AI agents and programming.")
print(result.final_output)

Sure, here it goes:

Why don't AI agents tell secrets?

Because they're afraid the programmers might overhear them in code review!


C:\Users\thiru\AppData\Local\Temp\ipykernel_31000\1789025027.py:1: RuntimeWarning: coroutine 'Runner.run' was never awaited
  result = await Runner.run(agent, "Tell me a joke about AI agents and programming.")


In [13]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell me a joke about AI agents and programming.")
    print(result.final_output)

Why don't AI agents tell secrets?

Because they're always worried about data breaches. 

And, why don't programmers like nature?

Because it has too many bugs!


## Sales Agent

In [15]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content

In [16]:
load_dotenv(override=True)

True

### Step 1 : Agent Workflow

In [17]:
instructions1 = """
You are a sales agent working for CompAI\
a company that provides SaaS tool for ensuring SOC2 compiliance and preparing for audits, powered by AI \
You write professional, serious cold email
"""
instructions2 = """
You are a humorous , encaging sales agent working for CompAI\
a company that provides SaaS tool for ensuring SOC2 compiliance and preparing for audits, powered by AI \
You write witty, funny cold email to engage the reader and make them want to learn more about the product
"""

instructions3 = """
you are a busy sales agent working for CompAI\
a company that provides SaaS tool for ensuring SOC2 compiliance and preparing for audits, powered by AI \
You write short, concise cold email that quickly gets to the point and respects the reader's time
"""

In [18]:
sales_agent1 = Agent(name="SalesAgent1", instructions=instructions1, model="gpt-4o-mini")
sales_agent2 = Agent(name="SalesAgent2", instructions=instructions2, model="gpt-4o-mini")
sales_agent3 = Agent(name="SalesAgent3", instructions=instructions3, model="gpt-4o-mini")    

In [21]:
result = Runner.run_streamed(sales_agent1, "Write a cold email to a potential customer about CompAI's SOC2 compliance tool.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Streamline Your SOC 2 Compliance Process with CompAI

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I represent CompAI, a leading provider of SaaS solutions designed to simplify and enhance SOC 2 compliance processes.

In today’s regulatory landscape, ensuring the security and confidentiality of your data is more crucial than ever. Our AI-powered tool streamlines the rigorous requirements of SOC 2 compliance, helping organizations like yours to efficiently prepare for audits and maintain ongoing compliance with ease.

Key benefits of our solution include:

- **Automated Checklist Management:** Stay organized with a dynamic checklist tailored to your unique compliance requirements.
- **Real-time Documentation:** Effortlessly gather and manage necessary documentation, ensuring you’re always audit-ready.
- **AI-Driven Insights:** Gain actionable insights to continuously improve your security posture and compliance readiness.

We underst

[non-fatal] Tracing: server error 504, retrying.


In [23]:
message = "Write a cold email to a potential customer about CompAI's SOC2 compliance tool."
with trace("parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
outputs = [result.final_output for result in results]

for output in outputs:
    print("----")
    print(output)
    print("\n")
    

----
Subject: Streamline Your SOC2 Compliance with CompAI’s AI-Powered Solution

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I am reaching out on behalf of CompAIa, where we specialize in advanced SaaS solutions designed to simplify and enhance the SOC2 compliance process.

Navigating the complexities of SOC2 compliance can be daunting and time-consuming, often diverting valuable resources away from your core business objectives. At CompAIa, we understand these challenges and have developed an AI-powered tool that not only streamlines the preparation for audits but also helps ensure ongoing compliance.

Key benefits of our solution include:

- **Automated Documentation**: Reduce manual effort and improve accuracy with AI-driven document generation and management.
- **Real-Time Monitoring**: Keep track of compliance metrics effortlessly, enabling proactive adjustments before audits.
- **Scalable Solutions**: Whether you're a startup or an es

In [26]:
sales_picker = Agent(
    name="SalesPicker",
    instructions="""
    You pick the best cold sales email from the given options\
    imagine you are a customer and the pick the one you are most likely to respond to \
    do not give an explanation, reply with the selected email only
    """,
    model="gpt-4o-mini"
)

In [28]:
message = "Write a cold email to a potential customer about CompAI's SOC2 compliance tool."

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for result in results]
    
    emails = "Cold sale Emails: \n\n" + "\n\n".join(outputs)

    best_email = await Runner.run(sales_picker, emails) 
    print(f"Best email:\n {best_email.final_output}")

Best email:
 Subject: 🚀 Are You Ready to Make SOC2 Compliance as Easy as Ordering Pizza? 🍕 

Hey [Recipient's Name],

Hope this email finds you well and not wrestling with a pile of compliance documents like they’re a mountain of dirty laundry! 

I'm reaching out from CompAIa, the friendly folks who believe getting SOC2 compliant should feel less like solving a Rubik's cube blindfolded and more like a smooth Sunday morning. 🌞 

You see, we’ve developed a magical little SaaS tool powered by AI that’s essentially like having a compliance wizard on your team (without the pointy hat or the questionable potions). It’ll help you breeze through SOC2 audits while keeping your sanity intact! 🧙‍♂️ 

Imagine this: no more frantic late-night meetings with your team, no more endless emails asking “Did we forget to document that thing again?” and certainly no more mid-night snack runs fueled by stressed-out compliance managers! With our tool, you’ll get:

1. **Auto-Generated Reports:** Like a coffee

#### Tool 

In [40]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-4o-mini")
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model="gpt-4o-mini")
sales_agent3 = Agent(name="Busy Sales Agent", instructions=instructions3, model="gpt-4o-mini")


In [41]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='\nYou are a sales agent working for CompAIa company that provides SaaS tool for ensuring SOC2 compiliance and preparing for audits, powered by AI You write professional, serious cold email\n', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [47]:
@function_tool
def send_email(body:str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
    from_email = Email("engjegant@gmail.com")
    to_email = To("engjegant@gmail.com")
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content)
    response = sg.client.mail.send.post(request_body=mail)
    return {"status":"success"}

In [48]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002637CB18550>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [49]:
tool1 = sales_agent1.as_tool(tool_name="sales_email_generator", tool_description="Generates a cold sales email to potential customers about CompAI's SOC2 compliance tool.")
tool1

FunctionTool(name='sales_email_generator', description="Generates a cold sales email to potential customers about CompAI's SOC2 compliance tool.", params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002637CC38F50>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [50]:
description = "Write a cold email to a potential customer about CompAI's SOC2 compliance tool."
tool1 = sales_agent1.as_tool(tool_name="sales_email_generator_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_generator_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_generator_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_email_generator_1', description="Write a cold email to a potential customer about CompAI's SOC2 compliance tool.", params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002637CC39C50>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='sales_email_generator_2', description="Write a cold email to a potential customer about CompAI's SOC2 compliance tool.", params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}

In [51]:
instructions = """
You are a sales agent working for CompAI\
You use the tools given to you to write and send cold emails to potential customers about CompAI's SOC2 compliance tool.\
You never generate sales emails on your own; you always use the tools\
you try all 3 sales_agent tools once before choosing the best one to send out using the send_email tool\
you pick the single best email that is most likely to get a response from the customer\
use the send_email tool to send the best email (and only the best email) to the user
"""
sales_manager = Agent(name="SalesManager", instructions=instructions, model="gpt-4o-mini", tools=tools)

message = "Send a cold sales email addressed to 'Dear CEO' about CompAI's SOC2 compliance tool."

with trace("Sales Manager Agent"):
    result = await Runner.run(sales_manager, message)
    print(result.final_output)

It seems there was an issue while trying to send the email. You can check the email draft below and send it yourself if you're ready:

---

**Subject:** Elevate Your Compliance with CompAI's SOC2 Solution

**Dear CEO,**

I hope this message finds you well. In today’s digital landscape, achieving and maintaining compliance with industry standards is not just a goal—it's essential for safeguarding your organization’s reputation and success.

At CompAI, we understand the complexities of SOC2 compliance, which is why we've developed an AI-powered SaaS tool that streamlines your compliance processes. Our solution offers:

- **Automated Documentation:** Reducing the manual effort typically required for compliance.
- **Continuous Monitoring:** Ensuring you are always audit-ready.
- **Enhanced Risk Management:** Identifying potential vulnerabilities before they escalate.

Let’s connect to discuss how CompAI can simplify your journey to SOC2 compliance and bolster your operational efficiency.



[non-fatal] Tracing: server error 504, retrying.


#### Handoff

In [52]:
subject_instructions = """
You can write a subject for a cold sales email. \
you are given a message and you need to write a subject for an email that is likely to get responses from potential customers. \
"""
html_instructions = """
You can convert a text email body to an HTML email body. \
you are given a text email body might have some markdown \
and you need to convert it to an HTML eamil body with simple, clear, complelling layout design
"""

subject_writer = Agent(name="SubjectWriter", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Writes compelling email subjects for cold sales emails.")

html_converter = Agent(name="HTMLConverter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Converts text email bodies to HTML format for better formatting and presentation.")

In [53]:
@function_tool
def send_html_email(subject:str, html_body:str)->Dict[str, str]:
    """ Send out an HTML email with the given subject and body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("engjegant@gmail.com")
    to_email = To("massjegajegan143@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status":"success"}

In [54]:
tools = [subject_tool, html_tool, send_html_email]

In [55]:
instructions = """
You are email formatter and sender. You receive the body of an email to be sent.\
You first use the subject_writer tool to write a subject or the email, then use the html_converter tool to convert the email body to HTML format, then use the send_html_email tool to send out the email with the generated subject and HTML body.\
"""

emailer_agent = Agent(
    name="EmailerAgent",
    instructions=instructions, 
    model="gpt-4o-mini", 
    tools=tools,
    handoff_description="Convert an Email to HTML and send it out"
    )


In [56]:
tools = [tool1 , tool2, tool3]
handoffs = [emailer_agent]


In [59]:
sales_manager_instructions = """
You are an sales manager for CompAI. You use the tools given to you to generate cold sales emails. \
you never generate sales emails yourself; you always use tools \
you try all 3 sales agent tools at least once before choosing the best one\
you can use the tools multiple times if youre not satisfied with the results from the first try\
you select the single best email using your  own judgement of which email is most likely to get a response from the customer\
After picking the best email, you handoff to the email manager agent to format and send the email
"""
sales_manager = Agent(
    name="SalesManager",
    instructions=sales_manager_instructions, 
    model="gpt-4o-mini", 
    tools=tools,
    handoffs=handoffs,
    handoff_description="Pick the best email and handoff to the email formatter and sender agent"
)

message = "Send out a cold sales email addressed to dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message, max_turns=30)
    print(result.final_output)

The cold sales email has been successfully sent to the CEO. If you need further assistance or wish to send more emails, feel free to let me know!


## Multomodel Integration

In [68]:
from dotenv import load_dotenv
from openai import OpenAI, AsyncOpenAI
from agents import Agent, Runner, trace, function_tool,OpenAIChatCompletionsModel,input_guardrail,GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os 
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [61]:
load_dotenv(override=True)

True

In [62]:
openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")

In [63]:
instruction1 = """
You are a sales agent working for CompAI\
a company that provides Saas tool for ensuring SOC2 compilance and preparing for audits, powered by AI \
You write professional, serious cold email
"""

instruction2 = """
You are humorous, engaging slaes agent working for CompAI\
a company that provides a SaaS tool for ensuring SOC2 compilance and preparing for audits, powered by AI \
You write witty, funny cold email to engage the reader and make them want to learn more about the product
"""

instruction3 = """
you are a busy sales agent working for CompAI\
a company that provides a SaaS tool for ensuring SOC2 compilance and preparing for audits,
You write short, concise cold email that quickly gets to the point and respects the reader's time
"""

In [67]:
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OLLAMA_BASE_URL = "http://localhost:11434"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

GROQ_MODEL = "llama-3.3-70b-versatile"
GEMINI_MODEL = "gemini-2.5-flash"
OLLAMA_MODEL = "gpt-oss:120b-cloud"

In [69]:
groq_client = AsyncOpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL)
ollama_client = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
gemini_client = AsyncOpenAI(api_key=gemini_api_key, base_url=GEMINI_BASE_URL)   

groq_model = OpenAIChatCompletionsModel(model= GROQ_MODEL, openai_client=groq_client)
ollama_model = OpenAIChatCompletionsModel(model= OLLAMA_MODEL, openai_client=ollama_client)
gemini_model = OpenAIChatCompletionsModel(model= GEMINI_MODEL, openai_client=gemini_client)


In [70]:
sales_agent1 = Agent(name="groq agent", instructions=instruction1, model=groq_model)
sales_agent2 = Agent(name="ollama agent", instructions=instruction2, model=ollama_model)
sales_agent3 = Agent(name="gemini agent", instructions=instruction3, model=gemini_model)

In [71]:
description = "Write a cold email to a potential customer about CompAI's SOC2 compliance tool."

tool1 = sales_agent1.as_tool(tool_name="sales_email_generator_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_generator_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_generator_3", tool_description=description)

In [72]:
@function_tool
def send_email(subject:str, html_body:str)->Dict[str, str]:
    """ Send out an HTML email with the given subject and body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
    from_email = Email("engjegant@gmail.com")
    to_email = Email("massjegajegan143@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content)
    response = sg.client.mail.send.post(request_body=mail.get())
    return {"status":"success"}

In [73]:
subject_instructions = """You can write a subject for a cold sales email. \
you are given a message and you need to write a subject for an email that is likely to get responses from potential customers. \
"""
html_instructions = """
You can convert a text email body to an HTML email body. \
you are given a text email body might have some markdown \
and you need to convert it to an HTML eamil body with simple, clear, complelling layout design
"""
subject_writer = Agent(name="SubjectWriter", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Writes compelling email subjects for cold sales emails.")

html_converter = Agent(name="HTMLConverter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter", tool_description="Converts text email bodies to HTML format for better formatting and presentation.")    

In [74]:
email_tools = [subject_tool, html_tool, send_email]

In [75]:
instructions = """
You are email formatter and sender. You receive the body of an email to be sent.\
You first use the subject_writer tool to write a subject for the email, then use the html
converter tool to convert the email body to HTML format, then use the send_email tool to send out the email with the generated subject and HTML body.\
"""
emailer_agent = Agent(
    name="EmailerAgent",
    instructions=instructions, 
    model="gpt-4o-mini", 
    tools=email_tools,
    handoff_description="Convert an Email to HTML and send it out"
    )

In [76]:
tools = [tool1 , tool2, tool3]
handoffs = [emailer_agent]

In [82]:
sales_manager_instructions = """
You are a sales manager for CompAI. You use the tools given to you to generate cold sales emails. 
You never generate sales emails yourself; you always use the tools.
To provide a fast response, try the sales agent tools exactly once to generate an email. Do not retry or loop.
Once you have generated the emails, immediately select the single best email using your own judgement.
After picking the best email, immediately handoff to the EmailerAgent to format and send the email.
"""
sales_manager = Agent(
    name="SalesManager",
    instructions=sales_manager_instructions, 
    model="gpt-4o-mini", 
    tools=tools,
    handoffs=handoffs,
    handoff_description="Pick the best email and handoff to the email formatter and sender agent"
)

In [84]:
message = "Send out a cold sales email addressed to dear CEO from Alice. The recipient email is xyz@example.com."
# Removed the tracing block to avoid the 504 server timeout error
result = await Runner.run(sales_manager, message, max_turns=30)
print(result.final_output)

MaxTurnsExceeded: Max turns (30) exceeded

In [ ]:
class NameCheckerOutput(BaseModel):
    is_name_in_message: bool
    name:str

guardrail_agent = Agent(
    name="GuardrailAgent",
    instructions="Check if the user is including someone's name in what they want you to do",
    output_type=NameCheckerOutput,
    model="gpt-4o-mini"
)

In [ ]:
@input_guardrail
async def guardrail_agent_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)
